# RadarNet-1D: FPGA Hackathon 2026 Training Notebook

**Hardware-Accelerated Edge AI for RF Anomaly Detection**

This notebook runs the complete RadarNet ML pipeline on Google Colab with GPU:
1. Upload preprocessed data (`data/` folder)
2. Train RadarNet-1D (ResNet-1D, ~29K params, 60 epochs)
3. Quantize to INT8 with BN folding
4. Export weights as `.hex` for Verilog BRAM
5. Generate testbench vectors

**Instructions:**
1. Go to **Runtime > Change runtime type > GPU (T4)**
2. Run all cells with **Runtime > Run all**
3. Download results from the `checkpoints/` and `weights/` folders when done

## 0. Setup & Dependencies

In [ ]:
!pip install torch numpy h5py scikit-learn matplotlib tqdm -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Upload Preprocessed Data

**Option A (Google Drive):** Upload your `data/` folder to Google Drive, then mount it below.

**Option B (Direct upload):** Use the file browser on the left to upload `X_train.npy`, `y_train.npy`, `X_val.npy`, `y_val.npy`, `X_test.npy`, `y_test.npy` into a `data/` folder.

Run the cell below for **Option A** (Google Drive). Skip if using direct upload.

In [ ]:
# === Option A: Mount Google Drive ===
# Upload your data/ folder to Google Drive first, then run this cell.
# Update DRIVE_DATA_PATH to point to your data folder in Drive.

USE_GOOGLE_DRIVE = True  # Set to False if uploading files directly

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Change this path to where your data/ folder is in Google Drive
    DRIVE_DATA_PATH = '/content/drive/MyDrive/FPGA-Hackathon/data'
    !cp -r "{DRIVE_DATA_PATH}" /content/data
    print("Data copied from Google Drive.")
else:
    import os
    os.makedirs('/content/data', exist_ok=True)
    print("Upload your .npy files to /content/data/ using the file browser.")

In [ ]:
# Verify data files exist
import os
import numpy as np

DATA_DIR = '/content/data'
required = ['X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy', 'X_test.npy', 'y_test.npy']
for f in required:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        arr = np.load(path)
        print(f"  {f:15s} shape={str(arr.shape):20s} dtype={arr.dtype}")
    else:
        print(f"  MISSING: {f}")
        raise FileNotFoundError(f"Missing {path} — please upload your data files.")

## 2. Model Definition (RadarNet-1D)

In [ ]:
import torch
import torch.nn as nn


class ResidualBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3,
                 stride: int = 1):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                               stride=stride, padding=padding, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                               stride=1, padding=padding, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class RadarNet1D(nn.Module):
    def __init__(self, in_channels: int = 2, num_classes: int = 4):
        super().__init__()
        self.conv_in = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=7, stride=1,
                      padding=3, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
        )
        self.conv_down = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=3, stride=2,
                      padding=1, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
        )
        self.block1 = ResidualBlock(32, 32, kernel_size=3, stride=1)
        self.block2 = ResidualBlock(32, 64, kernel_size=3, stride=2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv_in(x)
        x = self.conv_down(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.gap(x)
        x = x.squeeze(-1)
        x = self.fc(x)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Quick test
model = RadarNet1D(in_channels=2, num_classes=4)
print(f"RadarNet-1D  |  Parameters: {count_parameters(model):,}")
dummy = torch.randn(4, 2, 128)
out = model(dummy)
print(f"Input: {dummy.shape} -> Output: {out.shape}")

## 3. Training

In [ ]:
import json
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, precision_recall_fscore_support,
)

# ── Config ──
DEFENSE_LABELS = ["Normal", "Jammer", "Spoofer", "Interference"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 60
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 10

OUT_DIR = '/content/checkpoints'
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Device: {DEVICE}")

# ── Load data ──
def load_npy(data_dir):
    splits = {}
    for split in ("train", "val", "test"):
        X = np.load(os.path.join(data_dir, f"X_{split}.npy"))
        y = np.load(os.path.join(data_dir, f"y_{split}.npy"))
        X = np.transpose(X, (0, 2, 1)).astype(np.float32)
        y = y.astype(np.int64)
        splits[split] = (torch.from_numpy(X), torch.from_numpy(y))
    return splits

def make_loader(X, y, batch_size, shuffle=True):
    return DataLoader(
        TensorDataset(X, y), batch_size=batch_size,
        shuffle=shuffle, num_workers=2, pin_memory=(DEVICE == "cuda"),
    )

splits = load_npy(DATA_DIR)
train_loader = make_loader(*splits["train"], BATCH_SIZE, shuffle=True)
val_loader   = make_loader(*splits["val"],   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(*splits["test"],  BATCH_SIZE, shuffle=False)

print(f"Train: {splits['train'][0].shape[0]:,}  Val: {splits['val'][0].shape[0]:,}  Test: {splits['test'][0].shape[0]:,}")

In [ ]:
# ── Training functions ──
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(yb.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return total_loss / total, correct / total, all_preds, all_labels

In [ ]:
# ── Train ──
model = RadarNet1D(in_channels=2, num_classes=4).to(DEVICE)
print(f"RadarNet-1D  |  Parameters: {count_parameters(model):,}  |  Device: {DEVICE}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
patience_counter = 0
best_path = os.path.join(OUT_DIR, "radarnet_best.pth")

print(f"\n{'Epoch':>5}  {'TrLoss':>8}  {'TrAcc':>7}  {'VaLoss':>8}  "
      f"{'VaAcc':>7}  {'LR':>10}  {'Time':>6}")
print("\u2500" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]["lr"]
    print(f"{epoch:5d}  {tr_loss:8.4f}  {tr_acc:6.2%}  {va_loss:8.4f}  "
          f"{va_acc:6.2%}  {lr:10.2e}  {elapsed:5.1f}s")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), best_path)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break

print(f"\nBest validation accuracy: {best_val_acc:.2%}")
print(f"Checkpoint saved: {best_path}")

## 4. Evaluation on Test Set

In [ ]:
# Load best checkpoint and evaluate
model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
te_loss, te_acc, te_preds, te_labels = evaluate(model, test_loader, criterion, DEVICE)
print(f"Test accuracy: {te_acc:.2%}  |  Test loss: {te_loss:.4f}\n")

report = classification_report(te_labels, te_preds, target_names=DEFENSE_LABELS, digits=4)
print(report)

# Confusion matrix
cm = confusion_matrix(te_labels, te_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=DEFENSE_LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("RadarNet-1D Confusion Matrix (Test Set)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

# Training curves
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"], label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss Curves")
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(epochs_range, history["train_acc"], label="Train")
ax2.plot(epochs_range, history["val_acc"], label="Val")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_title("Accuracy Curves")
ax2.legend(); ax2.grid(True, alpha=0.3)
fig.suptitle("RadarNet-1D Training Progress", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "training_curves.png"), dpi=150)
plt.show()

# Save metrics
p, r, f1, _ = precision_recall_fscore_support(te_labels, te_preds, average=None, labels=[0,1,2,3])
metrics = {
    "best_val_acc": best_val_acc,
    "test_acc": te_acc,
    "test_loss": te_loss,
    "per_class": {
        DEFENSE_LABELS[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f1[i])}
        for i in range(4)
    },
    "epochs_trained": len(history["train_loss"]),
    "total_params": count_parameters(model),
}
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {OUT_DIR}/metrics.json")

## 5. INT8 Quantization with BN Folding

In [ ]:
import copy

# ── BN Folding ──
def fold_bn_into_conv(conv, bn):
    gamma  = bn.weight.data
    beta   = bn.bias.data
    mu     = bn.running_mean
    sigma  = torch.sqrt(bn.running_var + bn.eps)
    W = conv.weight.data.clone()
    b = conv.bias.data.clone() if conv.bias is not None else torch.zeros(W.size(0))
    scale = (gamma / sigma).reshape(-1, 1, 1)
    W_folded = W * scale
    b_folded = gamma * (b - mu) / sigma + beta
    folded_conv = nn.Conv1d(
        conv.in_channels, conv.out_channels, conv.kernel_size[0],
        stride=conv.stride[0], padding=conv.padding[0], bias=True,
    )
    folded_conv.weight.data = W_folded
    folded_conv.bias.data   = b_folded
    return folded_conv

def fold_all_bn(model):
    model = copy.deepcopy(model)
    model.eval()
    def _fold_sequential(seq):
        new_modules = []
        i = 0
        modules = list(seq.children())
        while i < len(modules):
            m = modules[i]
            if isinstance(m, nn.Conv1d) and i + 1 < len(modules) \
                    and isinstance(modules[i + 1], nn.BatchNorm1d):
                folded = fold_bn_into_conv(m, modules[i + 1])
                new_modules.append(folded)
                i += 2
            else:
                new_modules.append(m)
                i += 1
        return nn.Sequential(*new_modules)
    model.conv_in   = _fold_sequential(model.conv_in)
    model.conv_down = _fold_sequential(model.conv_down)
    for block in [model.block1, model.block2]:
        block.conv1 = fold_bn_into_conv(block.conv1, block.bn1)
        block.bn1   = nn.Identity()
        block.conv2 = fold_bn_into_conv(block.conv2, block.bn2)
        block.bn2   = nn.Identity()
        if isinstance(block.shortcut, nn.Sequential):
            block.shortcut = _fold_sequential(block.shortcut)
    return model

# ── INT8 Quantization ──
def symmetric_quantise_tensor(t, bits=8):
    qmin = -(2 ** (bits - 1))
    qmax = (2 ** (bits - 1)) - 1
    abs_max = t.abs().max().item()
    if abs_max == 0:
        return torch.zeros_like(t, dtype=torch.int8), 1.0
    scale = abs_max / qmax
    t_q = torch.clamp(torch.round(t / scale), qmin, qmax).to(torch.int8)
    return t_q, scale

def quantise_model_weights(model):
    quant_params = {}
    for name, param in model.named_parameters():
        t_q, scale = symmetric_quantise_tensor(param.data, bits=8)
        quant_params[name] = {"int8": t_q, "scale": scale}
    return quant_params

@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb).argmax(1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
    return correct / total

print("Functions defined. Running quantization...")

In [ ]:
# Run quantization pipeline
device = DEVICE

# Load best model
model_q = RadarNet1D(in_channels=2, num_classes=4)
model_q.load_state_dict(torch.load(best_path, map_location="cpu", weights_only=True))
model_q.to(device)

# Reload test data for quantization eval
X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))
X_test_t = np.transpose(X_test, (0, 2, 1)).astype(np.float32)
quant_test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test_t),
                  torch.from_numpy(y_test.astype(np.int64))),
    batch_size=BATCH_SIZE, shuffle=False,
)

# FP32 baseline
fp32_acc = evaluate_accuracy(model_q, quant_test_loader, device)
print(f"FP32 test accuracy: {fp32_acc:.4%}")

# Fold BN
print("\nFolding BatchNorm into convolution weights...")
folded_model = fold_all_bn(model_q)
folded_model.to(device)
folded_acc = evaluate_accuracy(folded_model, quant_test_loader, device)
print(f"BN-folded test accuracy: {folded_acc:.4%}  (delta = {folded_acc - fp32_acc:+.4%})")

# Quantize to INT8
print("\nQuantising all parameters to INT8 (symmetric)...")
quant_params = quantise_model_weights(folded_model)

# Verify INT8 accuracy
quant_model = copy.deepcopy(folded_model)
for name, param in quant_model.named_parameters():
    qp = quant_params[name]
    param.data = qp["int8"].float() * qp["scale"]
quant_model.to(device)
quant_acc = evaluate_accuracy(quant_model, quant_test_loader, device)
drop = fp32_acc - quant_acc
print(f"INT8 test accuracy: {quant_acc:.4%}  (delta from FP32 = {-drop:+.4%})")
status = "PASS" if drop < 0.01 else "FAIL"
print(f"\nQuantisation accuracy drop: {drop:.4%}  [{status}: target < 1%]")

# Save quantized artifacts
save_dict = {}
for name in quant_params:
    save_dict[name + ".int8"]  = quant_params[name]["int8"]
    save_dict[name + ".scale"] = torch.tensor(quant_params[name]["scale"])

quant_path = os.path.join(OUT_DIR, "radarnet_int8.pth")
torch.save(save_dict, quant_path)
print(f"\nINT8 quantised weights -> {quant_path}")

folded_path = os.path.join(OUT_DIR, "radarnet_folded.pth")
torch.save(folded_model.state_dict(), folded_path)
print(f"BN-folded FP32 model  -> {folded_path}")

total_bytes = sum(qp["int8"].numel() for qp in quant_params.values())
print(f"\nTotal INT8 weight bytes: {total_bytes:,}  ({total_bytes / 1024:.1f} KB)")

## 6. Export Weights as .hex for Verilog BRAM

In [ ]:
WEIGHTS_DIR = '/content/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

def int8_to_hex(val):
    if val < 0:
        val = val + 256
    return f"{val:02X}"

def export_tensor_hex(tensor, filepath):
    flat = tensor.flatten().numpy().astype(np.int8)
    with open(filepath, "w") as f:
        for val in flat:
            f.write(int8_to_hex(int(val)) + "\n")
    return len(flat)

LAYER_NAMES = {
    "conv_in.0.weight":         "L1_conv_in",
    "conv_in.0.bias":           "L1_conv_in_bias",
    "conv_down.0.weight":       "L2_conv_down",
    "conv_down.0.bias":         "L2_conv_down_bias",
    "block1.conv1.weight":      "L3_res1_conv1",
    "block1.conv1.bias":        "L3_res1_conv1_bias",
    "block1.conv2.weight":      "L4_res1_conv2",
    "block1.conv2.bias":        "L4_res1_conv2_bias",
    "block2.conv1.weight":      "L5_res2_conv1",
    "block2.conv1.bias":        "L5_res2_conv1_bias",
    "block2.conv2.weight":      "L6_res2_conv2",
    "block2.conv2.bias":        "L6_res2_conv2_bias",
    "block2.shortcut.0.weight": "L5_res2_proj",
    "block2.shortcut.0.bias":   "L5_res2_proj_bias",
    "fc.weight":                "L8_fc",
    "fc.bias":                  "L8_fc_bias",
}

ckpt = torch.load(quant_path, map_location="cpu", weights_only=True)
int8_keys = sorted([k for k in ckpt if k.endswith(".int8")])
scale_keys = sorted([k for k in ckpt if k.endswith(".scale")])

print(f"Found {len(int8_keys)} INT8 parameter tensors\n")
total_bytes = 0

for key in int8_keys:
    param_name = key.replace(".int8", "")
    tensor = ckpt[key]
    scale = ckpt[param_name + ".scale"].item()
    clean = LAYER_NAMES.get(param_name, param_name.replace(".", "_"))
    hex_file = os.path.join(WEIGHTS_DIR, f"{clean}.hex")
    n = export_tensor_hex(tensor, hex_file)
    total_bytes += n
    shape_s = "x".join(str(s) for s in tensor.shape)
    print(f"  {clean:30s} shape={shape_s:20s} values={n:>6,} scale={scale:.6f}")

# Scale factors
scales_path = os.path.join(WEIGHTS_DIR, "scales.txt")
with open(scales_path, "w") as f:
    for key in scale_keys:
        pn = key.replace(".scale", "")
        cn = LAYER_NAMES.get(pn, pn.replace(".", "_"))
        f.write(f"{cn} {ckpt[key].item():.10f}\n")

# Address map
addr_map = os.path.join(WEIGHTS_DIR, "address_map.txt")
addr = 0
with open(addr_map, "w") as f:
    f.write(f"{'Layer':<30s} {'Start':>8s} {'End':>8s} {'Size':>8s}\n")
    for key in int8_keys:
        pn = key.replace(".int8", "")
        cn = LAYER_NAMES.get(pn, pn.replace(".", "_"))
        sz = ckpt[key].numel()
        f.write(f"{cn:<30s} {addr:>8d} {addr+sz-1:>8d} {sz:>8d}\n")
        addr += sz

print(f"\n  Total INT8 values: {total_bytes:,} ({total_bytes/1024:.1f} KB)")
print(f"  Scales  -> {scales_path}")
print(f"  AddrMap -> {addr_map}")

## 7. Generate Testbench Vectors

In [ ]:
TB_DIR = '/content/tb_vectors'
os.makedirs(TB_DIR, exist_ok=True)
NUM_VECTORS = 50
SEED = 42

# Load test data
X_test_raw = np.load(os.path.join(DATA_DIR, "X_test.npy"))   # (N,128,2)
y_test_raw = np.load(os.path.join(DATA_DIR, "y_test.npy"))

# Select balanced subset
rng = np.random.default_rng(SEED)
n_per_class = NUM_VECTORS // 4
indices = []
for c in range(4):
    cls_idx = np.where(y_test_raw == c)[0]
    chosen = rng.choice(cls_idx, min(n_per_class, len(cls_idx)), replace=False)
    indices.append(chosen)
remaining = NUM_VECTORS - sum(len(x) for x in indices)
if remaining > 0:
    all_idx = np.concatenate(indices)
    pool = np.setdiff1d(np.arange(len(y_test_raw)), all_idx)
    indices.append(rng.choice(pool, remaining, replace=False))
indices = np.concatenate(indices)
rng.shuffle(indices)
indices = indices[:NUM_VECTORS]

X_sel = X_test_raw[indices]
y_sel = y_test_raw[indices]

# Run quantized model on selected samples
# (quant_model already loaded from step 5)
X_torch = torch.from_numpy(np.transpose(X_sel, (0, 2, 1)).astype(np.float32))
with torch.no_grad():
    logits = quant_model.to("cpu")(X_torch)
    preds = logits.argmax(1).numpy()

def float_to_int8_hex(val):
    ival = int(round(val * 127.0))
    ival = max(-128, min(127, ival))
    if ival < 0:
        ival += 256
    return f"{ival:02X}"

# Export input vectors
input_hex = os.path.join(TB_DIR, "input_vectors.hex")
with open(input_hex, "w") as f:
    for i in range(len(X_sel)):
        for t in range(128):
            i_val = float_to_int8_hex(X_sel[i, t, 0])
            q_val = float_to_int8_hex(X_sel[i, t, 1])
            f.write(f"{i_val}{q_val}\n")
print(f"Input vectors ({len(X_sel)} x 128 x 2) -> {input_hex}")

# Export expected labels
labels_hex = os.path.join(TB_DIR, "expected_labels.hex")
with open(labels_hex, "w") as f:
    for p in preds:
        f.write(f"{int(p):02X}\n")
print(f"Expected labels -> {labels_hex}")

# Export true labels
true_hex = os.path.join(TB_DIR, "true_labels.hex")
with open(true_hex, "w") as f:
    for l in y_sel:
        f.write(f"{int(l):02X}\n")
print(f"True labels     -> {true_hex}")

match = (preds == y_sel).sum()
print(f"\nQuantised model: {match}/{len(y_sel)} match true labels")
print(f"  Class distribution: {[int((y_sel==c).sum()) for c in range(4)]}")

## 8. Download Results

Run the cell below to zip all outputs and download them.

In [ ]:
import shutil

# Zip all results
shutil.make_archive('/content/radarnet_results', 'zip', '/content', 'checkpoints')
shutil.make_archive('/content/radarnet_weights', 'zip', '/content', 'weights')
shutil.make_archive('/content/radarnet_tb_vectors', 'zip', '/content', 'tb_vectors')

print("Archives created:")
print("  /content/radarnet_results.zip     (checkpoints + metrics + figures)")
print("  /content/radarnet_weights.zip     (.hex files for Verilog BRAM)")
print("  /content/radarnet_tb_vectors.zip  (testbench vectors)")

# Auto-download in Colab
try:
    from google.colab import files
    files.download('/content/radarnet_results.zip')
    files.download('/content/radarnet_weights.zip')
    files.download('/content/radarnet_tb_vectors.zip')
except ImportError:
    print("\nNot running in Colab — download the .zip files manually.")

In [ ]:
# Optionally save results back to Google Drive
if USE_GOOGLE_DRIVE:
    DRIVE_OUT = '/content/drive/MyDrive/FPGA-Hackathon/results'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    shutil.copytree('/content/checkpoints', os.path.join(DRIVE_OUT, 'checkpoints'), dirs_exist_ok=True)
    shutil.copytree('/content/weights', os.path.join(DRIVE_OUT, 'weights'), dirs_exist_ok=True)
    shutil.copytree('/content/tb_vectors', os.path.join(DRIVE_OUT, 'tb_vectors'), dirs_exist_ok=True)
    print(f"All results saved to Google Drive: {DRIVE_OUT}")
else:
    print("Skipping Google Drive save (USE_GOOGLE_DRIVE = False)")